In [2]:
!pip install faker

import pandas as pd
import numpy as np
from faker import Faker
import random

fake = Faker()
Faker.seed(42)
np.random.seed(42)
random.seed(42)

# 1. Base lists for messy categories
domains_messy = [
    'web dev', 'Web Development', 'WEB DEV', 'webdev',
    'data science', 'Data Science', 'datascience',
    'AI/ML', 'ai / ml', 'aiml',
    'Cloud Computing', 'cloud computing', 'cloud',
    'Cybersecurity', 'cyber security'
]

universities_list = ['NUST', 'FAST', 'LUMS', 'COMSATS', 'UET', 'GIKI', np.nan]
statuses_list = ['Under Review', 'Selected', 'Rejected', 'selected', 'SELECTED', 'rejected', np.nan]

records = []

# 2. Generate 82 distinct applicants with deliberate messy data
for i in range(82):
    # Name issues (some missing, some with extra whitespace)
    if i in [10, 25]:
        name = np.nan
    elif i in [5, 15, 30]:
        name = f"  {fake.name()}  "
    else:
        name = fake.name()

    # Email issues (some missing, some casing inconsistency)
    if i in [12, 40, 60]:
        email = np.nan
    elif i in [7, 18]:
        email = fake.email().upper()
    else:
        email = fake.email().lower()

    # Phone issues (stored as text with dashes/spaces)
    if i % 2 == 0:
        phone = f"0301-{fake.numerify('#######')}"
    else:
        phone = f"0302 {fake.numerify('#######')}"

    # Inconsistent domains
    domain = random.choice(domains_messy)

    # Inconsistent university (including missing values)
    uni = random.choice(universities_list)

    # Inconsistent date formats and corrupt dates
    if i == 50:
        date = "invalid_date"
    elif i % 2 == 0:
        date = fake.date_between(start_date='-60d', end_date='today').strftime('%Y-%m-%d')
    else:
        date = fake.date_between(start_date='-60d', end_date='today').strftime('%d/%m/%Y')

    # Status with missing and case inconsistencies
    status = random.choice(statuses_list)

    records.append({
        'Applicant Name': name,
        'Email': email,
        'Phone': phone,
        'Domain Applied': domain,
        'University': uni,
        'Application Date': date,
        'Status': status
    })

# Convert to DataFrame
df_mock = pd.DataFrame(records)

# 3. Add duplicates: 2 exact duplicates and 1 near-duplicate (same email, different name spacing)
exact_dupes = df_mock.iloc[[0, 1]].copy()
near_dupe = df_mock.iloc[[2]].copy()
near_dupe['Applicant Name'] = "  " + str(near_dupe['Applicant Name'].values[0]) + " "

df_mock = pd.concat([df_mock, exact_dupes, near_dupe], ignore_index=True)

# 4. Save to applicants.csv
df_mock.to_csv('applicants.csv', index=False)
print(f"Generated 'applicants.csv' successfully with {len(df_mock)} rows!")

Generated 'applicants.csv' successfully with 85 rows!


## Step 3: First Look — Auditing the Raw Dataset
Before modifying any data, we inspect its shape, column types, missing values, and duplicate rows to establish a baseline for our cleaning process.

In [3]:
# Load the raw dataset
df = pd.read_csv('applicants.csv')

print("--- Shape ---")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

print("\n--- First 5 Rows ---")
display(df.head())

print("\n--- Data Types & Non-Null Counts ---")
df.info()

print("\n--- Missing Values Per Column ---")
print(df.isnull().sum())

print(f"\n--- Exact Duplicate Rows: {df.duplicated().sum()} ---")

--- Shape ---
Rows: 85, Columns: 7

--- First 5 Rows ---


,Applicant Name,Email,Phone,Domain Applied,University,Application Date,Status
0,Allison Hill,donaldgarcia@example.net,0301-8196001,Cloud Computing,NUST,2026-09-02,Under Review
1,Alyssa Gonzalez,lrobinson@example.com,0302 0265423,cloud computing,LUMS,16/10/2026,Selected
2,Brandon Davis,cassandra07@example.net,0301-1618495,webdev,FAST,2026-09-23,rejected
3,Amber Perez,tracie31@example.com,0302 4752553,Web Development,GIKI,29/09/2026,rejected
4,Melinda Jones,amandasanchez@example.com,0301-6483503,cyber security,UET,2026-10-08,Under Review



--- Data Types & Non-Null Counts ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 85 entries, 0 to 84
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Applicant Name    83 non-null     object
 1   Email             82 non-null     object
 2   Phone             85 non-null     object
 3   Domain Applied    85 non-null     object
 4   University        71 non-null     object
 5   Application Date  85 non-null     object
 6   Status            79 non-null     object
dtypes: object(7)
memory usage: 4.8+ KB

--- Missing Values Per Column ---
Applicant Name       2
Email                3
Phone                0
Domain Applied       0
University          14
Application Date     0
Status               6
dtype: int64

--- Exact Duplicate Rows: 2 ---


### Summary of Initial Observations:
- **Dimensions:** The dataset contains 85 rows and 7 columns.
- **Missing Data:** Missing values are detected across `Applicant Name`, `Email`, `University`, and `Status`.
- **Duplicates:** Exact duplicate rows exist due to double submissions.
- **Data Types:** `Application Date` and `Phone` are stored as generic `object` (text) types instead of standard date and cleaned string formats.
- **Categorical Inconsistencies:** `Domain Applied` contains variations in casing, abbreviations, and spacing across the 5 domains.

## Step 4: Handle Missing Values (Column by Column, With Justification)
The treatment for missing data is tailored to the specific role of each column:
- **Applicant Name**: Critical identifier. A record without a name cannot be evaluated or attributed to a candidate; therefore, rows with missing names are dropped.
- **Email**: Critical contact key. Applicants cannot be contacted or scheduled without an email address; therefore, rows missing an email are dropped.
- **University**: Non-critical demographic field. Dropping entire records would discard valid candidates; missing values are imputed with `'Not Specified'`.
- **Status**: Procedural field. An unrecorded status implies the application has not yet been processed; missing values are imputed with the default `'Under Review'`.

In [4]:
# Record starting row count
initial_count = len(df)

# 1. Drop rows with missing critical identifiers (Applicant Name and Email)
df = df.dropna(subset=['Applicant Name', 'Email']).copy()

# 2. Fill non-critical fields with appropriate defaults
df['University'] = df['University'].fillna('Not Specified')
df['Status'] = df['Status'].fillna('Under Review')

print(f"Rows before handling missing values: {initial_count}")
print(f"Rows after dropping missing critical fields: {len(df)}")
print(f"Dropped rows: {initial_count - len(df)}")

print("\n--- Missing Values Remaining ---")
print(df.isnull().sum())

Rows before handling missing values: 85
Rows after dropping missing critical fields: 80
Dropped rows: 5

--- Missing Values Remaining ---
Applicant Name      0
Email               0
Phone               0
Domain Applied      0
University          0
Application Date    0
Status              0
dtype: int64


### Summary of Step 4 Actions:
- **Rows Processed:** Reduced the dataset from 85 rows to 80 rows by dropping 5 records that lacked essential identifying information (`Applicant Name` and `Email`).
- **Imputations Applied:** Filled all missing values in `University` with `'Not Specified'` and in `Status` with `'Under Review'`.
- **Outcome:** The dataset now has 0 remaining missing values across all columns.

## Step 5: Remove Exact and Near-Duplicate Applicants
Applicants occasionally submit multiple applications due to form resubmission or accidental double-clicking[cite: 3].
1. **Exact Duplicates:** We identify and drop rows where all columns match identically using `df.drop_duplicates()`[cite: 3].
2. **Near-Duplicates:** A candidate may re-apply with slight differences in name formatting or spacing. Because an applicant's `Email` is a more reliable unique identifier than their name, we normalize the email field and drop duplicate submissions sharing the same email, keeping only their first entry (`keep='first'`)[cite: 4].

In [5]:
# Check and drop exact duplicates
exact_dupes_count = df.duplicated().sum()
df = df.drop_duplicates().copy()

# Standardize Email casing and whitespace to catch near-duplicates
df['Email'] = df['Email'].str.strip().str.lower()

# Check and drop near-duplicates sharing the same email
near_dupes_count = df.duplicated(subset=['Email']).sum()
df = df.drop_duplicates(subset=['Email'], keep='first').copy()

print(f"Exact duplicates removed: {exact_dupes_count}")
print(f"Near-duplicates removed (by Email): {near_dupes_count}")
print(f"Total rows remaining after deduplication: {len(df)}")

Exact duplicates removed: 2
Near-duplicates removed (by Email): 1
Total rows remaining after deduplication: 77


### Summary of Step 5 Actions:
- **Exact Duplicates Removed:** Identified and eliminated 2 identical rows across all columns.
- **Near-Duplicates Removed:** Removed 1 redundant application identified by a matching email address (keeping the first submission).
- **Current Row Count:** 77 unique applicant records remain.

## Step 6: Standardize Text Fields
Inconsistent casing and spelling variations prevent accurate categorical analysis[cite: 4].
1. **Whitespace Cleaning:** We strip leading and trailing whitespace across all text columns[cite: 4].
2. **Casing Normalization:** We apply title case to `Applicant Name` and `Status` so variations like `"selected"` and `"SELECTED"` become unified[cite: 4].
3. **Domain Mapping:** We map informal track abbreviations and variations in `Domain Applied` into the 5 canonical track labels: **Web Development**, **Data Science**, **AI/ML**, **Cloud Computing**, and **Cybersecurity**[cite: 4].

In [6]:
# 1. Strip whitespace across string columns
text_cols = ['Applicant Name', 'Domain Applied', 'University', 'Status']
for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

# 2. Standardize casing
df['Applicant Name'] = df['Applicant Name'].str.title()
df['Status'] = df['Status'].str.title()

# 3. Define canonical mapping dictionary for Domain Applied
domain_mapping = {
    'web dev': 'Web Development',
    'webdev': 'Web Development',
    'web development': 'Web Development',
    'data science': 'Data Science',
    'datascience': 'Data Science',
    'ai/ml': 'AI/ML',
    'ai / ml': 'AI/ML',
    'aiml': 'AI/ML',
    'cloud computing': 'Cloud Computing',
    'cloud': 'Cloud Computing',
    'cybersecurity': 'Cybersecurity',
    'cyber security': 'Cybersecurity'
}

# Standardize domain names
df['Domain Applied'] = df['Domain Applied'].str.lower().map(domain_mapping)

# Verification check: Ensure exactly 5 clean categories exist
print("Unique values in 'Domain Applied':")
print(df['Domain Applied'].unique())
print(f"\nMissing values after mapping: {df['Domain Applied'].isnull().sum()}")

Unique values in 'Domain Applied':
['Cloud Computing' 'Web Development' 'Cybersecurity' 'AI/ML'
 'Data Science']

Missing values after mapping: 0


### Summary of Step 6 Actions:
- **Whitespace Cleaned:** Stripped invisible leading and trailing whitespace across all string columns[cite: 4].
- **Casing Normalized:** Standardized `Applicant Name` and `Status` to Title Case[cite: 4].
- **Domain Mapping Verified:** Successfully mapped all raw text variations into the 5 canonical domains with 0 unmapped values remaining[cite: 4, 16].

## Step 7: Fix Data Types (Application Date & Phone)
- **Application Date:** Stored as generic text with inconsistent formatting (`YYYY-MM-DD` vs `DD/MM/YYYY`) and potential corrupt entries[cite: 2, 5]. We convert it to native `datetime64[ns]` using `pd.to_datetime(..., errors='coerce')` so invalid dates gracefully become `NaT` and can be systematically dropped without crashing the notebook[cite: 5, 7].
- **Phone:** Punctuation (dashes and spaces) is stripped to make numbers consistent, but the column remains stored as text to preserve any leading zeros[cite: 2, 5].

In [7]:
# Convert Application Date using format='mixed' to handle both YYYY-MM-DD and DD/MM/YYYY
df['Application Date'] = pd.to_datetime(df['Application Date'], format='mixed', errors='coerce')

# Drop only genuinely corrupt dates (like "invalid_date")
invalid_dates_count = df['Application Date'].isnull().sum()
if invalid_dates_count > 0:
    df = df.dropna(subset=['Application Date']).copy()
    print(f"Dropped {invalid_dates_count} genuinely unparseable date row(s).")

# Clean phone numbers (strip dashes and spaces)
df['Phone'] = df['Phone'].astype(str).str.replace('-', '', regex=False).str.replace(' ', '', regex=False)

# Verify types and remaining rows
print("\n--- Verified Column Types ---")
df.info()
print(f"\nFinal row count: {len(df)}")

Dropped 1 genuinely unparseable date row(s).

--- Verified Column Types ---
<class 'pandas.core.frame.DataFrame'>
Index: 76 entries, 0 to 81
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Applicant Name    76 non-null     object        
 1   Email             76 non-null     object        
 2   Phone             76 non-null     object        
 3   Domain Applied    76 non-null     object        
 4   University        76 non-null     object        
 5   Application Date  76 non-null     datetime64[ns]
 6   Status            76 non-null     object        
dtypes: datetime64[ns](1), object(6)
memory usage: 4.8+ KB

Final row count: 76


# Step 8: Data Quality Summary

### 1. Starting Dataset
- **Initial dimensions:** 85 rows, 7 columns
- **Identified Issues:**
  - Missing entries in `Applicant Name`, `Email`, `University`, and `Status`[cite: 5]
  - 2 exact duplicate rows and 1 near-duplicate submission identified via matching email[cite: 5, 15]
  - Multiple casing, spacing, and abbreviation variants across the 5 domains in `Domain Applied`[cite: 4, 5]
  - `Application Date` stored as generic text with mixed date formats and unparseable strings[cite: 2, 5]
  - `Phone` numbers containing non-standard hyphens and space separators[cite: 2, 5]

### 2. Actions Taken
- Dropped 5 rows lacking essential identification (`Applicant Name` and `Email`)[cite: 3, 6, 14]
- Imputed missing `University` values with `'Not Specified'` and missing `Status` entries with `'Under Review'`[cite: 3, 6]
- Removed 2 exact duplicates and 1 near-duplicate record based on normalized email[cite: 3, 4, 6, 15]
- Standardized string casing and mapped all `Domain Applied` variations to the 5 official tracks[cite: 4, 6]
- Converted `Application Date` to native `datetime64[ns]` using mixed parsing and dropped 1 corrupt date row[cite: 5, 6, 20]
- Cleaned `Phone` number formats into uniform digit strings while maintaining string type to protect leading zeros[cite: 5, 6]

### 3. Final Dataset
- **Final dimensions:** 76 clean rows, 7 columns[cite: 6, 20]
- **Verification:** 0 missing values in critical fields, 0 duplicates, and verified types[cite: 6, 20].

In [8]:
# Step 9: Export clean dataset without an unnamed index column
df.to_csv('applicants_cleaned.csv', index=False)
print("applicants_cleaned.csv successfully exported!")

applicants_cleaned.csv successfully exported!
